# 05. 순서 피처 점검 (2단계 선별 모델 후보)
KAMP는 생산 순서의 인접 상관이 매우 높다(전류 0.96, 가압력 0.90, 동저항 0.92). 느린 변화(전극 마모 등)를 빼고 순간 이탈만 보는
`dev_*`(직전 20타점 중앙값 대비 편차), `dl_*`(직전 타점 대비 변화량) 피처를 만든다.
Stellantis N.V.는 순서를 파일 순서(차체 → 타점 번호)로 **가정**했고 위치 평균을 먼저 뺐다. 순서 가정이 틀리면 이 검증은 의미가 약하다는 한계가 있다.
통과 기준은 02의 기준 2·3(포착률, 같은 위치 안)을 그대로 쓴다.

In [1]:
import sys, time, numpy as np, pandas as pd, torch
sys.path.insert(0, ".")
from joblib import Parallel, delayed
from wl import runner, bags
pd.set_option("display.width", 250)
MODELS = ["const_rate", "seq_rule", "iforest", "autoencoder", "A_dllp_mlp", "B_count_mlp", "D_pseudo_logreg", "UB_logreg", "UB_catboost"]
CONDS = list(bags.CONDITIONS); SEEDS = list(range(5)); N_GPU = torch.cuda.device_count()
name = "seq_features"
def _task(i, m, c, s): return runner.run_one(m, c, s, feature_set="seq", gpu=i % max(N_GPU, 1))
if (runner.RES / f"{name}.pkl").exists():
    runs = runner.load_runs(name)
else:
    tasks = [(m, c, s) for m in MODELS for c in CONDS for s in SEEDS]
    t = time.time(); runs = Parallel(n_jobs=8, verbose=2)(delayed(_task)(i, *k) for i, k in enumerate(tasks))
    print(f"{len(runs)}개 실행 {time.time()-t:.0f}초"); runner.save_runs(runs, name)
by = {}
for r in runs: by.setdefault((r["model"], r["cond"]), []).append(r)
boots = dict(Parallel(n_jobs=16)(delayed(lambda k: (k, runner.bootstrap(by[k], n_boot=200)))(k) for k in by))
def fmt(m, c, k):
    r = boots[(m, c)].loc[k]; return f"{r['mean']:.3f} [{r['lo']:.3f}, {r['hi']:.3f}]"
KEYS = ["auroc", "wl_auroc", "recall@0.1", "wl_recall@0.1", "recall@5x"]
for c in ("orig", "k8_1"):
    print(f"\n=== seq | {c} ==="); display(pd.DataFrame({k: {m: fmt(m, c, k) for m in MODELS} for k in KEYS}))

[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.


[Parallel(n_jobs=8)]: Done  25 tasks      | elapsed:    2.5s


C:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[Parallel(n_jobs=8)]: Done 162 tasks      | elapsed:  1.9min


[Parallel(n_jobs=8)]: Done 180 out of 180 | elapsed:  2.1min finished


180개 실행 125초



=== seq | orig ===


,auroc,wl_auroc,recall@0.1,wl_recall@0.1,recall@5x
const_rate,"0.500 [0.500, 0.500]","0.500 [0.500, 0.500]","0.100 [0.100, 0.100]","0.100 [0.100, 0.100]","0.196 [0.192, 0.202]"
seq_rule,"0.494 [0.419, 0.545]","0.449 [0.401, 0.517]","0.152 [0.086, 0.212]","0.073 [0.045, 0.121]","0.223 [0.160, 0.279]"
iforest,"0.516 [0.439, 0.574]","0.468 [0.420, 0.542]","0.162 [0.083, 0.225]","0.071 [0.048, 0.132]","0.233 [0.170, 0.297]"
autoencoder,"0.550 [0.501, 0.614]","0.524 [0.462, 0.603]","0.144 [0.096, 0.225]","0.113 [0.072, 0.164]","0.218 [0.145, 0.313]"
A_dllp_mlp,"0.512 [0.459, 0.569]","0.543 [0.461, 0.600]","0.076 [0.040, 0.107]","0.095 [0.071, 0.134]","0.179 [0.116, 0.251]"
B_count_mlp,"0.501 [0.449, 0.554]","0.536 [0.462, 0.613]","0.071 [0.033, 0.118]","0.113 [0.068, 0.166]","0.182 [0.103, 0.262]"
D_pseudo_logreg,"0.457 [0.388, 0.530]","0.438 [0.355, 0.534]","0.089 [0.032, 0.152]","0.094 [0.048, 0.140]","0.175 [0.108, 0.247]"
UB_logreg,"0.515 [0.447, 0.601]","0.557 [0.479, 0.660]","0.089 [0.046, 0.148]","0.099 [0.075, 0.132]","0.226 [0.135, 0.329]"
UB_catboost,"0.598 [0.551, 0.659]","0.467 [0.393, 0.554]","0.166 [0.116, 0.237]","0.063 [0.044, 0.105]","0.261 [0.195, 0.357]"



=== seq | k8_1 ===


,auroc,wl_auroc,recall@0.1,wl_recall@0.1,recall@5x
const_rate,"0.500 [0.500, 0.500]","0.500 [0.500, 0.500]","0.100 [0.100, 0.100]","0.100 [0.100, 0.100]","0.049 [0.047, 0.050]"
seq_rule,"0.500 [0.419, 0.571]","0.494 [0.406, 0.616]","0.158 [0.075, 0.236]","0.111 [0.055, 0.196]","0.132 [0.047, 0.215]"
iforest,"0.527 [0.451, 0.601]","0.523 [0.453, 0.621]","0.168 [0.083, 0.257]","0.111 [0.060, 0.200]","0.137 [0.049, 0.214]"
autoencoder,"0.596 [0.542, 0.680]","0.564 [0.488, 0.649]","0.221 [0.126, 0.317]","0.128 [0.061, 0.202]","0.063 [0.000, 0.162]"
A_dllp_mlp,"0.500 [0.428, 0.571]","0.502 [0.394, 0.588]","0.095 [0.032, 0.168]","0.091 [0.046, 0.160]","0.042 [0.011, 0.100]"
B_count_mlp,"0.463 [0.397, 0.539]","0.455 [0.352, 0.535]","0.063 [0.019, 0.161]","0.084 [0.040, 0.126]","0.063 [0.012, 0.147]"
D_pseudo_logreg,"0.451 [0.375, 0.557]","0.391 [0.284, 0.522]","0.116 [0.053, 0.186]","0.111 [0.049, 0.158]","0.063 [0.000, 0.092]"
UB_logreg,"0.528 [0.467, 0.587]","0.562 [0.454, 0.665]","0.105 [0.033, 0.161]","0.114 [0.070, 0.156]","0.074 [0.018, 0.129]"
UB_catboost,"0.545 [0.485, 0.602]","0.450 [0.382, 0.535]","0.158 [0.080, 0.229]","0.036 [0.022, 0.107]","0.021 [0.000, 0.049]"


In [2]:
rows = []
for m in MODELS[1:]:
    row = {"모델": m}
    for c in CONDS:
        b = boots[(m, c)]
        row[c] = ("O" if b.loc["recall@0.1", "lo"] > .10 else "X") + ("O" if (b.loc["wl_recall@0.1", "lo"] > .10 and b.loc["wl_auroc", "lo"] > .5) else "X")
    rows.append(row)
print("[기준2 포착률][기준3 같은 위치 안]"); pd.DataFrame(rows)

[기준2 포착률][기준3 같은 위치 안]


,모델,orig,k8_4,k8_1,k8_05
0,seq_rule,XX,XX,XX,XX
1,iforest,XX,XX,XX,XX
2,autoencoder,XX,XX,OX,XX
3,A_dllp_mlp,XX,XX,XX,XX
4,B_count_mlp,XX,XX,XX,XX
5,D_pseudo_logreg,XX,XX,XX,XX
6,UB_logreg,XX,XX,XX,XX
7,UB_catboost,OX,OX,XX,XX
